Compute exploratory clinical panel contrasts and mechanism summaries.....


In [1]:
from pathlib import Path
import os
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()))
if os.name == 'nt' and (not str(PROJECT_ROOT).startswith('\\\\?\\')):
    PROJECT_ROOT = Path('\\\\?\\' + str(PROJECT_ROOT))


In [2]:
import glob
import io
import json
import os
import re
import sys
import warnings
import numpy as np
import pandas as pd

def read_csv_normalized(*args, **kwargs):
    data = pd.read_csv(*args, **kwargs)
    if 'Vignette' in data.columns:
        data['Vignette'] = data['Vignette'].map(lambda value: value.replace('\r\n', '\n') if isinstance(value, str) else value)
    return data
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings('ignore')
ROOT = str(PROJECT_ROOT)
OUT = os.path.normpath(os.path.join(ROOT, 'analyses/protocol_design/exploratory_panel_comparisons'))
os.makedirs(OUT, exist_ok=True)
ALPHA = 0.05
PBASE = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
PBASE_NON = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_truncated'
NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
SINGLES = ['gpt', 'qwen', 'deepseekv4']
EXP = 'experiments/panel_parameter_sweeps'
REG = []

def log(family, hypothesis, test, n, effect, p, note=''):
    REG.append({'семейство': family, 'гипотеза': hypothesis, 'тест': test, 'n': n, 'эффект': effect, 'p': p, 'примечание': note})

def read(rel):
    p = os.path.normpath(os.path.join(ROOT, rel))
    if not os.path.exists(p):
        return None
    d = read_csv_normalized(p, sep=';', encoding='utf-8-sig', low_memory=False)
    if 'table_source' in d.columns:
        d = d[d['table_source'].fillna('result_df') == 'result_df']
    if 'Judge_score' not in d.columns:
        return None
    d = d.dropna(subset=['Judge_score', 'Vignette']).drop_duplicates('Vignette', keep='first')
    return d.set_index('Vignette')

def wilc(a, b):
    ix = a.index.intersection(b.index)
    d = a.loc[ix].astype(float) - b.loc[ix].astype(float)
    p = stats.wilcoxon(d)[1] if (d != 0).any() else np.nan
    return (len(ix), float(d.mean()), float((d > 0).mean()), float(p))
SWEEPS = {'температура': (EXP + '/temperature/llm_as_judge/delphi_temperature_*_judge.csv', 'temperature_([0-9.]+)_judge'), 'число экспертов': (EXP + '/experts/llm_as_judge/delphi_experts_*_judge.csv', 'group\\d+_(\\d+)models'), 'пунктов опросника': (EXP + '/questionnaire_items/llm_as_judge/delphi_questionnaire_items_*_judge.csv', 'items_(\\d+)_judge'), 'вопросов на эксперта': (EXP + '/questions_per_expert/llm_as_judge/delphi_questions_per_expert_*_judge.csv', 'expert_(\\d+)_judge')}

def sweep_matrix(pattern, rx):
    cols = {}
    for f in sorted(glob.glob(os.path.normpath(os.path.join(ROOT, pattern)))):
        m = re.search(rx, os.path.basename(f))
        if not m:
            continue
        d = read(os.path.relpath(f, ROOT))
        if d is None:
            continue
        lvl = float(m.group(1))
        cols.setdefault(lvl, []).append(d['Judge_score'].astype(float))
    if not cols:
        return None
    ser = {lvl: pd.concat(v, axis=1).mean(axis=1) for lvl, v in cols.items()}
    full = pd.DataFrame(ser)
    cover = full.notna().sum()
    keep = cover[cover >= 0.9 * cover.max()].index
    M = full[sorted(keep)].dropna()
    return (M, cover.sort_index())

def s_sweeps():
    rows, detail = ([], [])
    for name, (pat, rx) in SWEEPS.items():
        got = sweep_matrix(pat, rx)
        if got is None:
            continue
        M, cover = got
        if M.shape[1] < 3:
            continue
        if cover.min() < cover.max():
            rho_c, p_c = stats.spearmanr(np.array(cover.index, float), cover.values)
            log('S. развёртки', 'доля успешно оценённых кейсов падает с ростом «%s»' % name, 'Spearman по уровням', len(cover), float(rho_c), float(p_c))
        lv = np.array(M.columns, float)
        fr = stats.friedmanchisquare(*[M[c].values for c in M.columns])
        log('S. развёртки', '%s влияет на Judge_score' % name, 'Фридман (парный омнибус, %d уровней)' % M.shape[1], len(M), float(fr.statistic), float(fr.pvalue))
        rhos = [stats.spearmanr(lv, M.loc[i].values).statistic for i in M.index]
        rhos = np.array([r for r in rhos if np.isfinite(r)])
        tw = stats.wilcoxon(rhos)
        log('S. развёртки', 'монотонный тренд по параметру «%s»' % name, 'Уилкоксон по покейсным rho', len(rhos), float(np.mean(rhos)), float(tw.pvalue))
        mu = M.mean()
        best, worst = (mu.idxmax(), mu.idxmin())
        n, d, sh, p = wilc(M[best], M[worst])
        log('S. развёртки', 'лучший уровень «%s» лучше худшего' % name, 'Уилкоксон, %.4g против %.4g' % (best, worst), n, d, p, 'размах отобран по данным -> p оптимистичен')
        rows.append({'развёртка': name, 'уровней в анализе': M.shape[1], 'уровней всего': len(cover), 'кейсов': len(M), 'min': mu.min(), 'max': mu.max(), 'размах': mu.max() - mu.min(), 'лучший уровень': best, 'худший уровень': worst, 'Фридман p': fr.pvalue, 'средний rho': np.mean(rhos), 'p тренда': tw.pvalue})
        for l in cover.index:
            detail.append({'развёртка': name, 'уровень': l, 'оценено кейсов': int(cover[l]), 'Judge': mu.get(l, np.nan), 'SD по кейсам': M[l].std() if l in M.columns else np.nan, 'в парном анализе': l in M.columns})
    return (pd.DataFrame(rows).set_index('развёртка'), pd.DataFrame(detail))

def s_early_stop():
    a = read(EXP + '/rounds/llm_as_judge/delphi_optimum_early_stop_judge.csv')
    b = read(EXP + '/rounds/llm_as_judge/delphi_rounds_no_early_stop_judge.csv')
    if a is None or b is None:
        return None
    n, d, sh, p = wilc(b['Judge_score'], a['Judge_score'])
    log('S. развёртки', 'принудительные 3 раунда лучше ранней остановки', 'Уилкоксон, парно по 60 виньеткам', n, d, p)
    ix = a.index.intersection(b.index)
    r = pd.to_numeric(a.loc[ix, 'delphi_rounds_executed'], errors='coerce')
    sub = []
    for k in (1, 2, 3):
        s = ix[r == k]
        if len(s) < 5:
            continue
        nn, dd, ss, pp = wilc(b.loc[s, 'Judge_score'], a.loc[s, 'Judge_score'])
        sub.append({'остановилась на раунде': k, 'n': nn, 'Δ форс−ранняя': dd, 'доля>': ss, 'p': pp})
        log('S. развёртки', 'форсирование помогает там, где ранняя остановка сработала на R%d' % k, 'Уилкоксон', nn, dd, pp)
    return (pd.DataFrame({'плечо': ['ранняя остановка', 'принудительные 3 раунда'], 'Judge': [a['Judge_score'].mean(), b['Judge_score'].mean()], 'n': [len(a), len(b)]}).set_index('плечо'), pd.DataFrame(sub))

def s_experts_nested():
    chain = [('group2_4models', 4), ('group3_5models', 5), ('group7_6models', 6), ('group5_7models', 7), ('group1_10models', 10)]
    cols = {}
    for tag, k in chain:
        d = read(EXP + '/experts/llm_as_judge/delphi_experts_%s_judge.csv' % tag)
        if d is not None:
            cols[k] = d['Judge_score'].astype(float)
    M = pd.DataFrame(cols).dropna()
    if M.shape[1] < 3:
        return None
    lv = np.array(M.columns, float)
    fr = stats.friedmanchisquare(*[M[c].values for c in M.columns])
    log('S. развёртки', 'число экспертов влияет на Judge_score (ВЛОЖЕННАЯ цепочка)', 'Фридман, %d вложенных панелей' % M.shape[1], len(M), float(fr.statistic), float(fr.pvalue))
    rhos = np.array([stats.spearmanr(lv, M.loc[i].values).statistic for i in M.index])
    rhos = rhos[np.isfinite(rhos)]
    tw = stats.wilcoxon(rhos)
    log('S. развёртки', 'монотонный рост качества с числом экспертов (ВЛОЖЕННАЯ цепочка)', 'Уилкоксон по покейсным rho', len(rhos), float(np.mean(rhos)), float(tw.pvalue))
    n, d, sh, pmax = wilc(M[max(M.columns)], M[min(M.columns)])
    log('S. развёртки', '10 экспертов лучше 4 (ВЛОЖЕННАЯ цепочка, края заданы заранее)', 'Уилкоксон', n, d, pmax)
    tab = pd.DataFrame({'Judge': M.mean(), 'SD по кейсам': M.std()})
    tab.index.name = 'число экспертов (вложенно)'
    return (tab, fr.pvalue, float(np.mean(rhos)), tw.pvalue)

def s_variance():
    got = sweep_matrix(*SWEEPS['температура'])
    if got is None:
        return None
    M, _ = got
    lv = np.array(M.columns, float)
    sd = M.std().values
    mu = M.mean().values
    r1, p1 = stats.spearmanr(lv, sd)
    r2, p2 = stats.spearmanr(lv, mu)
    log('S. развёртки', 'температура увеличивает РАЗБРОС качества между кейсами', 'Spearman по уровням', len(lv), float(r1), float(p1))
    log('S. развёртки', 'температура двигает СРЕДНИЙ балл', 'Spearman по уровням', len(lv), float(r2), float(p2))
    dev = M.sub(M.median(axis=1), axis=0).abs()
    rhos = np.array([stats.spearmanr(lv, dev.loc[i].values).statistic for i in dev.index])
    rhos = rhos[np.isfinite(rhos)]
    tw = stats.wilcoxon(rhos)
    log('S. развёртки', 'покейсное отклонение от собственной медианы растёт с температурой', 'Уилкоксон по покейсным rho', len(rhos), float(np.mean(rhos)), float(tw.pvalue))
    return pd.DataFrame({'температура': lv, 'средний Judge': mu, 'SD между кейсами': sd})

def tost_nulls():

    def t(x, bound=0.05):
        x = np.asarray(x, float)
        se = x.std(ddof=1) / np.sqrt(len(x))
        if se == 0:
            return 0.0
        df = len(x) - 1
        return float(max(stats.t.sf((x.mean() + bound) / se, df), stats.t.cdf((x.mean() - bound) / se, df)))
    rows = []
    a = read(EXP + '/rounds/llm_as_judge/delphi_optimum_early_stop_judge.csv')
    b = read(EXP + '/rounds/llm_as_judge/delphi_rounds_no_early_stop_judge.csv')
    if a is not None and b is not None:
        ix = a.index.intersection(b.index)
        d = b.loc[ix, 'Judge_score'].astype(float) - a.loc[ix, 'Judge_score'].astype(float)
        rows.append({'нулевая гипотеза': 'принудительные 3 раунда = ранняя остановка', 'n': len(d), 'эффект': d.mean(), 'TOST ±0.05': t(d.values)})
    for name in ['температура', 'пунктов опросника', 'вопросов на эксперта']:
        got = sweep_matrix(*SWEEPS[name])
        if got is None:
            continue
        M, _ = got
        d = M[max(M.columns)] - M[min(M.columns)]
        rows.append({'нулевая гипотеза': 'крайние уровни «%s» эквивалентны' % name, 'n': len(d), 'эффект': d.mean(), 'TOST ±0.05': t(d.values)})
    return pd.DataFrame(rows).set_index('нулевая гипотеза')

def c_covariates():
    dl = read(PBASE + '/llm_as_judge/delphi_base_noroles_judge.csv')
    b = {m: read(NB % (m, 'noroles', m)) for m in SINGLES}
    ix = dl.index
    for s in b.values():
        if s is not None:
            ix = ix.intersection(s.index)
    mx = pd.concat([b[m].loc[ix, 'Judge_score'].astype(float) for m in SINGLES], axis=1).max(axis=1)
    t = pd.DataFrame({'gap': dl.loc[ix, 'Judge_score'].astype(float) - mx, 'delphi': dl.loc[ix, 'Judge_score'].astype(float)})
    for c in ['Genealogy', 'Type', 'Age Group', 'Sub Clinical Specialties']:
        t[c] = dl.loc[ix, c]
    rows = []
    for c in ['Genealogy', 'Type', 'Age Group', 'Sub Clinical Specialties']:
        grp = [g['gap'].values for _, g in t.groupby(c) if len(g) >= 8]
        if len(grp) < 2:
            continue
        kw = stats.kruskal(*grp)
        log('C. ковариаты кейса', 'преимущество панели зависит от «%s»' % c, 'Kruskal-Wallis по группам', len(t), float(kw.statistic), float(kw.pvalue))
        for lvl, g in t.groupby(c):
            if len(g) < 8:
                continue
            w = stats.wilcoxon(g['gap'])
            rows.append({'ковариата': c, 'уровень': lvl, 'n': len(g), 'gap': g['gap'].mean(), 'p (gap≠0)': w.pvalue})
            log('C. ковариаты кейса', 'панель бьёт лучшую одиночную на «%s = %s»' % (c, lvl), 'Уилкоксон', len(g), float(g['gap'].mean()), float(w.pvalue))
        rows.append({'ковариата': c, 'уровень': '— Kruskal-Wallis —', 'n': len(t), 'gap': np.nan, 'p (gap≠0)': kw.pvalue})
    return pd.DataFrame(rows)

def p_self_ratings():
    rows = []
    for v, root in [('full', PBASE), ('non', PBASE_NON)]:
        d = read(root + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in d.columns if c.startswith('delphi_ratings_') and c.endswith('_mean')]
        for c in cols:
            x = pd.to_numeric(d[c], errors='coerce')
            ok = x.notna()
            if ok.sum() < 20:
                continue
            rho, p = stats.spearmanr(x[ok], d.loc[ok, 'Judge_score'].astype(float))
            short = c.replace('delphi_ratings_', '').replace('_mean', '')
            rows.append({'вариант': v, 'рейтинг': short, 'n': int(ok.sum()), 'rho с Judge_score': rho, 'p': p})
            log('P. самооценка панели', 'внутренний рейтинг «%s» предсказывает Judge_score' % short, 'Spearman', int(ok.sum()), float(rho), float(p), 'вариант ' + v)
    return pd.DataFrame(rows).set_index(['вариант', 'рейтинг'])

def v_disagreement():
    rows = []
    for v, root, r1 in [('full', PBASE, PBASE + '/llm_as_judge/round_evaluation/delphi_base_noroles_protocol_r1_judge.csv'), ('non', PBASE_NON, PBASE_NON + '/llm_as_judge/round_evaluation/delphi_base_noroles_protocol_r1_judge.csv')]:
        d = read(root + '/llm_as_judge/delphi_base_noroles_judge.csv')
        vc = [c for c in d.columns if c.startswith('r1_emergency__')]
        if not vc:
            continue
        sub = d[vc]
        nuniq = sub.apply(lambda r: r.dropna().nunique(), axis=1)
        valid = sub.notna().sum(axis=1)
        ok = valid == len(vc)
        y = d.loc[ok, 'Judge_score'].astype(float)
        x = nuniq[ok]
        if x.nunique() < 2 or len(y) < 20:
            continue
        rho, p = stats.spearmanr(x, y)
        rows.append({'вариант': v, 'показатель': 'число различных меток в R1', 'n': len(y), 'rho с Judge_score': rho, 'p': p})
        log('V. разногласие в R1', 'разногласие в R1 предсказывает качество финала', 'Spearman (только полные голосования)', len(y), float(rho), float(p), 'вариант ' + v)
        g = read(r1)
        if g is not None:
            ixx = y.index.intersection(g.index)
            gain = y.loc[ixx] - g.loc[ixx, 'Judge_score'].astype(float)
            rho2, p2 = stats.spearmanr(x.loc[ixx], gain)
            rows.append({'вариант': v, 'показатель': 'то же, но исход = прирост FIN−R1', 'n': len(ixx), 'rho с Judge_score': rho2, 'p': p2})
            log('V. разногласие в R1', 'разногласие в R1 предсказывает прирост FIN−R1', 'Spearman', len(ixx), float(rho2), float(p2), 'вариант ' + v)
    return pd.DataFrame(rows).set_index(['вариант', 'показатель'])

def r_reliability():

    def repeats(paths):
        m = [read(p) for p in paths]
        m = [x['Judge_score'].astype(float) for x in m if x is not None]
        return pd.concat(m, axis=1).dropna() if len(m) >= 3 else None
    P = repeats([PBASE + '/llm_as_judge/judge_robustness/delphi_base_noroles_retest_r%d_judge.csv' % r for r in range(1, 6)])
    rows, per_case = ([], {})
    if P is not None:
        per_case['P-base Delphi'] = P.std(axis=1)
        rows.append({'плечо': 'P-base Delphi', 'n кейсов': len(P), 'средний балл': P.mean(axis=1).mean(), 'SD внутри кейса': P.std(axis=1).mean(), 'SD между кейсами': P.mean(axis=1).std()})
    for m in SINGLES:
        B = repeats(['experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/judge_robustness/baseline_noroles_%s_retest_r%d_judge.csv' % (m, m, r) for r in range(1, 6)])
        if B is None:
            continue
        per_case['baseline ' + m] = B.std(axis=1)
        rows.append({'плечо': 'baseline ' + m, 'n кейсов': len(B), 'средний балл': B.mean(axis=1).mean(), 'SD внутри кейса': B.std(axis=1).mean(), 'SD между кейсами': B.mean(axis=1).std()})
    tab = pd.DataFrame(rows).set_index('плечо')
    tab['ICC-подобная доля'] = 1 - tab['SD внутри кейса'] ** 2 / (tab['SD внутри кейса'] ** 2 + tab['SD между кейсами'] ** 2)
    cmp_ = []
    if 'P-base Delphi' in per_case:
        for k, s in per_case.items():
            if k == 'P-base Delphi':
                continue
            ix = per_case['P-base Delphi'].index.intersection(s.index)
            d = per_case['P-base Delphi'].loc[ix] - s.loc[ix]
            p = stats.wilcoxon(d)[1] if (d != 0).any() else np.nan
            cmp_.append({'против': k, 'n': len(ix), 'Δ SD (панель − одиночная)': d.mean(), 'p': p})
            log('R. надёжность', 'панель СТАБИЛЬНЕЕ одиночной модели при повторах', 'Уилкоксон по покейсным SD, против ' + k, len(ix), float(d.mean()), float(p))
    return (tab, pd.DataFrame(cmp_).set_index('против'))

def j_second_judge():
    rows = []
    dl1 = read(PBASE_NON + '/llm_as_judge/delphi_base_noroles_judge.csv')
    dl2 = read(PBASE_NON + '/llm_as_judge/judge_gpt41/delphi_base_noroles_judge.csv')
    if dl2 is None:
        return None
    ix = dl1.index.intersection(dl2.index)
    rho, p = stats.spearmanr(dl1.loc[ix, 'Judge_score'].astype(float), dl2.loc[ix, 'Judge_score'].astype(float))
    rows.append({'плечо': 'P-base (non)', 'n': len(ix), 'судья 1': dl1.loc[ix, 'Judge_score'].mean(), 'судья 2 (gpt-4.1)': dl2.loc[ix, 'Judge_score'].mean(), 'rho между судьями': rho, 'p': p})
    log('J. второй судья', 'оценки двух судей согласованы (P-base)', 'Spearman', len(ix), float(rho), float(p))
    for m in SINGLES:
        b1 = read(NB % (m, 'noroles', m))
        b2 = read('experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/judge_gpt41/baseline_noroles_%s_judge.csv' % (m, m))
        if b1 is None or b2 is None:
            continue
        i2 = b1.index.intersection(b2.index)
        rho, p = stats.spearmanr(b1.loc[i2, 'Judge_score'].astype(float), b2.loc[i2, 'Judge_score'].astype(float))
        rows.append({'плечо': 'baseline ' + m, 'n': len(i2), 'судья 1': b1.loc[i2, 'Judge_score'].mean(), 'судья 2 (gpt-4.1)': b2.loc[i2, 'Judge_score'].mean(), 'rho между судьями': rho, 'p': p})
        log('J. второй судья', 'оценки двух судей согласованы (baseline %s)' % m, 'Spearman', len(i2), float(rho), float(p))
        n, d, sh, pp = wilc(dl2['Judge_score'], b2['Judge_score'])
        rows.append({'плечо': '  -> P-base − %s ПО СУДЬЕ 2' % m, 'n': n, 'судья 1': np.nan, 'судья 2 (gpt-4.1)': d, 'rho между судьями': sh, 'p': pp})
        log('J. второй судья', 'вывод «панель > одиночной» держится у судьи 2 (%s)' % m, 'Уилкоксон', n, d, pp)
    return pd.DataFrame(rows).set_index('плечо')

def a_decomposition():
    rows = []
    dl = read(PBASE + '/llm_as_judge/delphi_base_noroles_judge.csv')
    for m in SINGLES:
        b = read(NB % (m, 'noroles', m))
        if b is None:
            continue
        ix = dl.index.intersection(b.index)
        for col, label in [('judge_earned', 'заработано'), ('judge_penalties', 'штрафы')]:
            a = pd.to_numeric(dl.loc[ix, col], errors='coerce')
            c = pd.to_numeric(b.loc[ix, col], errors='coerce')
            mx = pd.to_numeric(dl.loc[ix, 'judge_max'], errors='coerce')
            d = ((a - c) / mx).dropna()
            p = stats.wilcoxon(d)[1] if (d != 0).any() else np.nan
            rows.append({'против': 'baseline ' + m, 'компонента': label, 'n': len(d), 'Δ (в долях judge_max)': d.mean(), 'p': p})
            log('A. декомпозиция балла', 'панель отличается от одиночной по компоненте «%s»' % label, 'Уилкоксон, против ' + m, len(d), float(d.mean()), float(p))
    return pd.DataFrame(rows).set_index(['против', 'компонента'])

def show(title, obj, note=''):
    print('\n' + '=' * 104)
    print(title)
    if note:
        print(note)
    print('=' * 104)
    if obj is None:
        print('(нет данных)')
    elif isinstance(obj, tuple):
        for o in obj:
            print(o.to_string() if hasattr(o, 'to_string') else o)
            print()
    else:
        print(obj.to_string() if hasattr(obj, 'to_string') else obj)

def main():
    print('=' * 104)
    print('СКРИНИНГ НОВЫХ ГИПОТЕЗ НА ДАННЫХ dataverse_gold  (статус: РАЗВЕДОЧНЫЙ)')
    print('=' * 104)
    sw, det = s_sweeps()
    show('S1. ПАРАМЕТРИЧЕСКИЕ РАЗВЁРТКИ ПРОТОКОЛА (delphi_exp, парно по 60 виньеткам)', sw, 'Фридман -- парный омнибус по всем уровням. Тренд -- Уилкоксон по покейсным rho.')
    show('S1b. ПОУРОВНЕВЫЕ СРЕДНИЕ', det.set_index(['развёртка', 'уровень']))
    det.to_csv(os.path.normpath(os.path.join(OUT, 'protocol_parameter_sweep_levels.csv')), index=False, encoding='utf-8-sig')
    nest = s_experts_nested()
    if nest is not None:
        tab, pfr, rho, ptr = nest
        show('S1c. ЧИСЛО ЭКСПЕРТОВ -- ВЛОЖЕННАЯ ЦЕПОЧКА (состав контролирован)', tab, 'Фридман p = %.3g | средний покейсный rho = %+.3f, p = %.3g' % (pfr, rho, ptr))
    var = s_variance()
    if var is not None:
        show('S1d. ТЕМПЕРАТУРА: СРЕДНЕЕ ПРОТИВ РАЗБРОСА', var.set_index('температура'))
    es = s_early_stop()
    show('S2. ПРЯМОЙ ЭКСПЕРИМЕНТ: РАННЯЯ ОСТАНОВКА ПРОТИВ ПРИНУДИТЕЛЬНЫХ 3 РАУНДОВ', es, 'Единственный в проекте НЕреконструированный тест гипотезы о раундах.')
    show('C. ЗАВИСИТ ЛИ ПРЕИМУЩЕСТВО ПАНЕЛИ ОТ СВОЙСТВ КЕЙСА (full, noroles, n=228)', c_covariates().set_index(['ковариата', 'уровень']), 'gap = Judge(P-base) − max(3 одиночные модели), покейсно.')
    show('P. ПРЕДСКАЗЫВАЮТ ЛИ ВНУТРЕННИЕ РЕЙТИНГИ ПАНЕЛИ КАЧЕСТВО ПРОТОКОЛА', p_self_ratings(), 'Рейтинги заполнены только там, где панель дошла до опросника.')
    show('V. ПРЕДСКАЗЫВАЕТ ЛИ РАЗНОГЛАСИЕ В R1 КАЧЕСТВО И ПРИРОСТ', v_disagreement())
    tab, cmp_ = r_reliability()
    show('R. НАДЁЖНОСТЬ ПРИ 5 ПОВТОРАХ', (tab, cmp_), 'SD внутри кейса -- разброс одного кейса по 5 прогонам. Меньше = стабильнее.')
    show('J. УСТОЙЧИВОСТЬ КО ВТОРОМУ СУДЬЕ (gpt-4.1)', j_second_judge(), 'ВНИМАНИЕ: судья 2 прогнан только на варианте non панели.')
    show('A. ДЕКОМПОЗИЦИЯ: ЗАРАБОТАННОЕ ПРОТИВ ШТРАФОВ', a_decomposition())
    show('T. КЛЮЧЕВЫЕ НУЛИ ЧЕРЕЗ ЭКВИВАЛЕНТНОСТЬ (TOST, граница ±0.05)', tost_nulls(), 'TOST < 0.05 => доказано, что эффект МЕНЬШЕ границы, а не просто «не найден».')
    r = pd.DataFrame(REG).dropna(subset=['p'])
    r['BH внутри семейства'] = False
    for fam, sub in r.groupby('семейство'):
        r.loc[sub.index, 'BH внутри семейства'] = multipletests(sub['p'], alpha=ALPHA, method='fdr_bh')[0]
    r['BH по всему экрану'] = multipletests(r['p'], alpha=ALPHA, method='fdr_bh')[0]
    r['Holm по всему экрану'] = multipletests(r['p'], alpha=ALPHA, method='holm')[0]
    r = r.sort_values('p')
    r.to_csv(os.path.normpath(os.path.join(OUT, 'exploratory_contrasts_with_multiple_comparison_corrections.csv')), index=False, encoding='utf-8-sig')
    summ = r.groupby('семейство').agg(тестов=('p', 'size'), сырых=('p', lambda s: int((s < ALPHA).sum())), BH_в_семействе=('BH внутри семейства', 'sum'), BH_общий=('BH по всему экрану', 'sum'), Holm=('Holm по всему экрану', 'sum'))
    show('ИТОГ: СКОЛЬКО ГИПОТЕЗ ВЫЖИЛО', summ)
    print('\nВсего тестов: %d | сырых p<0.05: %d | BH по экрану: %d | Holm: %d | ожидалось ложных при H0: %.1f' % (len(r), int((r['p'] < ALPHA).sum()), int(r['BH по всему экрану'].sum()), int(r['Holm по всему экрану'].sum()), ALPHA * len(r)))
    show('ВЫЖИВШИЕ ПОСЛЕ BH ПО ВСЕМУ ЭКРАНУ (кандидаты на подтверждающую проверку)', r[r['BH по всему экрану']][['семейство', 'гипотеза', 'тест', 'n', 'эффект', 'p', 'примечание']])
    print('\nCSV: %s' % OUT)
if __name__ == '__main__':
    main()


СКРИНИНГ НОВЫХ ГИПОТЕЗ НА ДАННЫХ dataverse_gold  (статус: РАЗВЕДОЧНЫЙ)



S1. ПАРАМЕТРИЧЕСКИЕ РАЗВЁРТКИ ПРОТОКОЛА (delphi_exp, парно по 60 виньеткам)
Фридман -- парный омнибус по всем уровням. Тренд -- Уилкоксон по покейсным rho.
                      уровней в анализе  уровней всего  кейсов       min       max    размах  лучший уровень  худший уровень  Фридман p  средний rho  p тренда
развёртка                                                                                                                                                     
температура                          17             20      60  0.791668  0.833703  0.042035             0.2             1.4   0.450910    -0.014676  0.443910
число экспертов                       5              5      60  0.760938  0.839045  0.078107             7.0             5.0   0.000003     0.222284  0.001247
пунктов опросника                    18             18      60  0.795795  0.840472  0.044677            10.0            15.0   0.348477    -0.026878  0.459481
вопросов на эксперта                 10         


S1c. ЧИСЛО ЭКСПЕРТОВ -- ВЛОЖЕННАЯ ЦЕПОЧКА (состав контролирован)
Фридман p = 0.226 | средний покейсный rho = +0.073, p = 0.32
                               Judge  SD по кейсам
число экспертов (вложенно)                        
4                           0.807110      0.118746
5                           0.827562      0.120348
6                           0.824892      0.106372
7                           0.839045      0.109358
10                          0.819808      0.117033



S1d. ТЕМПЕРАТУРА: СРЕДНЕЕ ПРОТИВ РАЗБРОСА
             средний Judge  SD между кейсами
температура                                 
0.1               0.812842          0.103856
0.2               0.833703          0.115450
0.3               0.811752          0.113717
0.4               0.806882          0.108757
0.5               0.811815          0.119885
0.6               0.813627          0.134256
0.7               0.808720          0.121822
0.8               0.814492          0.113975
0.9               0.820805          0.107420
1.0               0.825503          0.118332
1.1               0.818608          0.136859
1.2               0.825987          0.118758
1.3               0.802223          0.139616
1.4               0.791668          0.140766
1.5               0.795730          0.123398
1.6               0.795743          0.152470
1.7               0.808675          0.131667



S2. ПРЯМОЙ ЭКСПЕРИМЕНТ: РАННЯЯ ОСТАНОВКА ПРОТИВ ПРИНУДИТЕЛЬНЫХ 3 РАУНДОВ
Единственный в проекте НЕреконструированный тест гипотезы о раундах.
                            Judge   n
плечо                                
ранняя остановка         0.819368  60
принудительные 3 раунда  0.825800  60

   остановилась на раунде   n  Δ форс−ранняя     доля>         p
0                       1  28       0.020486  0.428571  0.868870
1                       2  22       0.003618  0.454545  0.872105
2                       3  10      -0.026730  0.200000  0.400814



...


V. ПРЕДСКАЗЫВАЕТ ЛИ РАЗНОГЛАСИЕ В R1 КАЧЕСТВО И ПРИРОСТ
                                            n  rho с Judge_score         p
вариант показатель                                                        
full    число различных меток в R1        227          -0.075311  0.258466
        то же, но исход = прирост FIN−R1  227          -0.038426  0.564639
non     число различных меток в R1        126           0.128107  0.152843
        то же, но исход = прирост FIN−R1  125           0.043441  0.630493



R. НАДЁЖНОСТЬ ПРИ 5 ПОВТОРАХ
SD внутри кейса -- разброс одного кейса по 5 прогонам. Меньше = стабильнее.
                     n кейсов  средний балл  SD внутри кейса  SD между кейсами  ICC-подобная доля
плечо                                                                                            
P-base Delphi              60      0.804561         0.077760          0.092929           0.588176
baseline gpt               60      0.638372         0.075644          0.149120           0.795339
baseline qwen              60      0.574930         0.082490          0.133067           0.722392
baseline deepseekv4        60      0.662181         0.079909          0.136099           0.743643

                      n  Δ SD (панель − одиночная)         p
против                                                      
baseline gpt         60                   0.002115  0.611488
baseline qwen        60                  -0.004730  0.531488
baseline deepseekv4  60                  -0.002149  0.819483



J. УСТОЙЧИВОСТЬ КО ВТОРОМУ СУДЬЕ (gpt-4.1)
ВНИМАНИЕ: судья 2 прогнан только на варианте non панели.
                                      n   судья 1  судья 2 (gpt-4.1)  rho между судьями             p
плечо                                                                                                
P-base (non)                         60  0.778975           0.734480           0.806030  7.897513e-15
baseline gpt                         60  0.648547           0.604988           0.777253  2.834294e-13
  -> P-base − gpt ПО СУДЬЕ 2         60       NaN           0.129492           0.766667  4.921289e-07
baseline qwen                        60  0.585405           0.528712           0.762949  1.387223e-12
  -> P-base − qwen ПО СУДЬЕ 2        60       NaN           0.205768           0.916667  3.927653e-10
baseline deepseekv4                  59  0.657976           0.651934           0.755399  4.724026e-12
  -> P-base − deepseekv4 ПО СУДЬЕ 2  59       NaN           0.078046           0.62


A. ДЕКОМПОЗИЦИЯ: ЗАРАБОТАННОЕ ПРОТИВ ШТРАФОВ
                                  n  Δ (в долях judge_max)             p
против              компонента                                          
baseline gpt        заработано  229               0.149044  2.497932e-31
                    штрафы      229              -0.015090  3.436872e-09
baseline qwen       заработано  229               0.211066  2.267347e-36
                    штрафы      229              -0.016653  9.120993e-09
baseline deepseekv4 заработано  228               0.139700  2.182193e-30
                    штрафы      228              -0.009516  1.648055e-06



T. КЛЮЧЕВЫЕ НУЛИ ЧЕРЕЗ ЭКВИВАЛЕНТНОСТЬ (TOST, граница ±0.05)
TOST < 0.05 => доказано, что эффект МЕНЬШЕ границы, а не просто «не найден».
                                                     n    эффект  TOST ±0.05
нулевая гипотеза                                                            
принудительные 3 раунда = ранняя остановка          60  0.006432    0.005244
крайние уровни «температура» эквивалентны           60 -0.004167    0.002581
крайние уровни «пунктов опросника» эквивалентны     60 -0.021988    0.022741
крайние уровни «вопросов на эксперта» эквивалентны  60 -0.033310    0.154720

ИТОГ: СКОЛЬКО ГИПОТЕЗ ВЫЖИЛО
                       тестов  сырых  BH_в_семействе  BH_общий  Holm
семейство                                                           
A. декомпозиция балла       6      6               6         6     6
C. ковариаты кейса         28     21              20        20    10
J. второй судья             7      7               7         7     6
P. самооценка панели    